# Week 5: Chat Template 对比 + Loss Masking 可视化

**目标**: 理解不同 chat template 的差异,可视化 loss masking 的效果。

In [ ]:
import torch
from transformers import AutoTokenizer
import matplotlib.pyplot as plt

## 1. 同一条对话,不同 Template 的对比

In [ ]:
CONVERSATION = [
    {"role": "system", "content": "You are a helpful medical assistant."},
    {"role": "user", "content": "What are the symptoms of diabetes?"},
    {"role": "assistant", "content": "Common symptoms include increased thirst, frequent urination, and fatigue."},
]

# 加载 tokenizer (如果某些模型需要 HF 登录,可以先跳过)
tokenizers = {}
for name, model_id in [
    ("Qwen", "Qwen/Qwen2.5-3B-Instruct"),
    # ("Llama-3", "meta-llama/Meta-Llama-3-8B-Instruct"),
]:
    try:
        tokenizers[name] = AutoTokenizer.from_pretrained(model_id)
    except Exception as e:
        print(f"{name} 加载失败: {e}")

In [ ]:
for name, tok in tokenizers.items():
    prompt = tok.apply_chat_template(CONVERSATION, tokenize=False, add_generation_prompt=True)
    tokens = tok.encode(prompt)
    print(f"\n{'='*60}")
    print(f"Model: {name}")
    print(f"Token count: {len(tokens)}")
    print(f"Preview:\n{prompt[:200]}...")
    print(f"\nTokens: {tokens[:20]}...")

## 2. 手动构造 ChatML 格式 (Qwen)

In [ ]:
# ChatML 格式
chatml = """
<|im_start|>system
You are a helpful medical assistant.<|im_end|>
<|im_start|>user
What are the symptoms of diabetes?<|im_end|>
<|im_start|>assistant
Common symptoms include increased thirst, frequent urination, and fatigue.<|im_end|>
""".strip()

print("ChatML 格式:")
print(chatml)

# 统计 special token 数量
im_start = chatml.count("<|im_start|>")
im_end = chatml.count("<|im_end|>")
print(f"\n<|im_start|>: {im_start} 次")
print(f"<|im_end|>: {im_end} 次")

## 3. Loss Masking 可视化

看看哪些 token 参与 loss 计算,哪些被 mask 掉了

In [ ]:
def visualize_loss_masking(tokenizer, conversation):
    """
    可视化: 哪些 token 参与 loss 计算 (assistant response),
    哪些被 mask 掉了 (prompt 部分 = -100)
    """
    prompt = tokenizer.apply_chat_template(conversation, tokenize=False)
    tokens = tokenizer.encode(prompt)
    
    # 简化的 masking 逻辑 (实际更复杂,需要精确匹配 assistant turn)
    labels = [-100] * len(tokens)  # 默认全部 mask
    
    # TODO: 实现精确的 mask_labels 逻辑
    # 找到 assistant 回复的起止位置,把这些位置的 labels 设为 token id
    
    # 可视化
    colors = ['green' if l != -100 else 'red' for l in labels]
    plt.figure(figsize=(16, 2))
    plt.bar(range(len(tokens)), [1]*len(tokens), color=colors, width=1)
    plt.xlabel("Token Index")
    plt.yticks([])
    plt.title("Loss Masking (Green=计算Loss, Red=Mask=-100)")
    plt.show()
    
    print(f"总 token 数: {len(tokens)}")
    print(f"参与 loss 的 token: {sum(1 for l in labels if l != -100)}")
    print(f"被 mask 的 token: {sum(1 for l in labels if l == -100)}")

if tokenizers:
    visualize_loss_masking(list(tokenizers.values())[0], CONVERSATION)

## 4. Template 错配实验

用错误的 template 训练会发生什么?

In [ ]:
# TODO: 实验设计
# 1. 用 Llama-3 template 格式化数据
# 2. 用这些数据 fine-tune Qwen 模型
# 3. 观察推理时模型是否会产生 Llama-3 的特殊 token (如 <|eot_id|>)
# 4. 对比正确 template 和错误 template 的效果差异

print("Template 错配会导致:")
print("1. 模型学到错误的 special token 分布")
print("2. 推理时产生不属于目标模型的 token")
print("3. 性能显著下降")

---

**交付物**:
- 3 种 template 的 token 数对比表
- Loss masking 可视化图
- `loss_masking.py` 中的 `mask_labels` 实现